# GitOps: how fast is CO2 rising?

You just joined a group that publishes one number: how fast CO2 at Mauna Loa is rising now, in ppm per year. Everything is in Git, so you expect to rerun it and get their number. You can't. The code won't run, and when it does, it doesn't give the number they published.

You'll fix both, and then you'll prove the fix by deleting the environment and the clone and getting the same number back. Keep one idea in mind. The document and the data are the pets, and everything that runs them is cattle.

**The short path takes about 15 minutes:** do step 1, then run its check. Steps 2 and 3 are optional and take about 15 more. Only reading? Each step shows the failure and says how to approach the fix.

## Get a terminal

You read this page and paste commands into a terminal. You don't run anything on the page. The output shown here is from a real run of these same commands on the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-01-gitops?urlpath=lab/workspaces/exercise).** It's free and there's nothing to install. The first start can take a few minutes. You get a terminal on the left, already in the `project` folder, and this page on the right. To copy a command, select it and press Ctrl+C, or Cmd+C on a Mac. To paste it in the terminal, press Ctrl+Shift+V, or Cmd+V on a Mac, or right-click and choose Paste. To edit a file, double-click it in the file panel on the far left, and save with Ctrl+S. A Binder session is temporary and has 2 GB of memory, so download anything you want to keep.

**Or use your laptop's terminal,** with this page open in your browser. It works on Linux and macOS, on x86_64 or ARM, and on Windows through WSL2. You need [conda](https://github.com/conda-forge/miniforge), and you don't need Jupyter. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`. Paste this once:

```bash
git clone --branch binder-01-gitops --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git gitops
cd gitops
conda env create --quiet --file .binder/environment.yml
conda activate repro-gitops
python scripts/setup_project.py
cd project
```

Every command below runs in that `project` folder. To start over, go to the folder above it and run `python scripts/setup_project.py --reset`, which deletes your work.

## The repository

`project` is the group's repository, with six commits by three people. There's also a stand-in for GitHub in `../remote`.

```bash
git log --format='%h  %an: %s'
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import run

run("git log --format='%h  %an: %s'")

$ git log --format='%h  %an: %s'
a06a448  Sam Okafor: Describe the project in the README
bbd77c3  Lee Tran: Read the CSV with pandas
805e276  Lee Tran: Tidy params.yaml
e358e01  Sam Okafor: Add a test of the fit and a holdout comparison
303c8f0  Priya Rao: Start the fit at 2000, and rerun
8c62c2e  Priya Rao: Add the Mauna Loa data and a first trend fit


## Step 1: make it run

The README says to run `python analyze.py`. Try it.

```bash
python analyze.py
```

In [2]:
# Recorded output. You don't need to run this.
run("python analyze.py")

$ python analyze.py
Traceback (most recent call last):
  File "~/project/analyze.py", line 4, in <module>
    import pandas as pd
ModuleNotFoundError: No module named 'pandas'


Someone installed pandas on their own laptop and started using it. It runs for them and for nobody else. The rule is simple. As you add packages, update the environment file.

Your turn:

1. Open `environment.yml` in the `project` folder and declare what's missing, on a new line in the same form as the others. Don't pin a version. That's the lock's job. Save the file.
2. Confirm the edit before you build anything. This should show your one added line. If it prints nothing, you edited another file or didn't save.

```bash
git diff environment.yml
```

3. Build the project's environment from that file, write the lock, and commit both. This takes a minute or two. `environment.yml` lists what you asked for, and the lock lists what you got. The lock is named after your platform, because each platform needs its own. The commit should say `2 files changed`.

```bash
conda env create --quiet --prefix ../envs/co2 --file environment.yml
conda list --prefix ../envs/co2 --explicit --md5 > explicit-$(conda config --show subdir | cut -d' ' -f2).txt
git add environment.yml explicit-*.txt
git commit -m "Declare pandas and lock the environment"
```

4. Run the analysis in that environment, look at what it did to `metrics.json`, and undo that for now. Don't commit `metrics.json` yet. Step 2 is about why.

```bash
../envs/co2/bin/python analyze.py
git diff metrics.json
git restore metrics.json
```

### Check the short path

Run this when step 1 is done. Each line says what to do next. When it passes, a stranger can build your environment from what's in Git and you've finished the short path. You can stop here.

```bash
python ../check.py --short
```

On the starter project it fails like this:

In [3]:
# Recorded output. You don't need to run this.
run("python ../check.py --short")

$ python ../check.py --short
FAIL  analyze.py imports pandas, and the committed environment.yml does not declare it
FAIL  No lock file is committed. Write one with conda list --explicit --md5 and commit it.
FAIL  No environment at envs/co2. Build it with the commands in step 1.

0 of 3 parts pass.


## Step 2 (optional): find out why the number moved

Here's what the group published, and what a rerun of the committed `params.yaml` and data gives. `check_fresh.py` has been in the repository since the first commit. If the code breaks, the document should break. Here the number stopped matching and nothing told anyone.

```bash
cat metrics.json
python check_fresh.py
```

In [4]:
# Recorded output. You don't need to run this.
run("cat metrics.json", "python check_fresh.py")

$ cat metrics.json
{
  "growth_rate_ppm_per_year": 2.681,
  "rmse_ppm": 0.473,
  "seasonal_amplitude_ppm": 6.625
}

$ python check_fresh.py
STALE: metrics.json does not match a rerun of params.yaml and the data.
  growth_rate_ppm_per_year: metrics.json says 2.681, a rerun gives 2.749
  rmse_ppm: metrics.json says 0.473, a rerun gives 0.469
  seasonal_amplitude_ppm: metrics.json says 6.625, a rerun gives 6.63


The tests pass anyway. They test the fit, not the committed number.

```bash
python -m pytest -q
```

In [5]:
# Recorded output. You don't need to run this.
run("python -m pytest -q")

$ python -m pytest -q
..                                                                       [100%]
2 passed in 0.03s


One commit changed something without a rerun. You could read all six. With 600 you wouldn't. `git bisect run` does a binary search through the history, using any command that exits 0 for good and nonzero for bad. `check_fresh.py` is that kind of command.

Your turn. Note the commit ID it reports, and read that commit with `git show`.

```bash
git bisect start HEAD $(git rev-list --max-parents=0 HEAD)
git bisect run python check_fresh.py
```

Then end the search.

```bash
git bisect reset
```

Now you have to decide, and no tool will do it for you. Should the group publish the old number or the rerun? A higher degree always fits the years it saw a little better, so that comparison can't settle it. `holdout.py` fits up to 2019 and scores each degree on 2020 onward, which the fit never saw. Read the table and pick a degree you can defend.

```bash
python holdout.py
```

Your turn:

1. Set `degree` in `params.yaml`, then rerun, stage, and confirm. `check_fresh.py` reads what you staged.

```bash
../envs/co2/bin/python analyze.py
git add params.yaml metrics.json
python check_fresh.py
```

2. Commit. In the message, name the commit `git bisect` found and say why you chose this degree. The next person will read that message instead of asking you.

```bash
git commit -m "REPLACE THIS with the commit ID bisect found, the degree you chose, and why"
```

3. The repository has shipped a hook since its first commit, and nobody installed it. Install it.

```bash
cp hooks/pre-commit .git/hooks/pre-commit
```

4. Try to commit a stale number. Change `degree` in `params.yaml`, save it, and paste this. The hook should refuse the commit, and the last two lines put `params.yaml` back.

```bash
git add params.yaml
git commit -m "Try to commit a stale number"
git restore --staged params.yaml
git restore params.yaml
```

## Step 3 (optional): delete everything that runs it

Your clone and your environment are cattle. If the project is right, you can throw them away. Push first, because only what's on the remote survives. At the end you tag the release, so people can find exactly this version later.

Your turn. `git status` near the end should print nothing, which means you got the same number. A clone doesn't carry hooks, so install yours again before you run the check. On Binder, JupyterLab may say `Directory not found` while `project` is gone. Close that message. It does no harm, and you can open `project` again in the file panel.

```bash
git push origin main
cd ..
rm -rf project envs/co2
git clone --config user.name="Workshop Learner" --config user.email="learner@example.org" remote/co2.git project
cd project
conda create --yes --quiet --prefix ../envs/co2 --file explicit-$(conda config --show subdir | cut -d' ' -f2).txt
../envs/co2/bin/python analyze.py
git status --short
git tag -a v1.0 -m "Publication version"
git push origin v1.0
```

## Check everything

This is the full check, for steps 1 to 3. It reads what you committed and tells you what to do next. It knows the right growth rate only as a salted digest, so reading `check.py` won't give you the number.

```bash
python ../check.py
```

On the starter project it fails like this:

In [6]:
# Recorded output. You don't need to run this.
run("python ../check.py")

$ python ../check.py
FAIL  analyze.py imports pandas, and the committed environment.yml does not declare it
FAIL  No lock file is committed. Write one with conda list --explicit --md5 and commit it.
FAIL  The committed metrics.json does not match a rerun. Run python check_fresh.py to see the numbers.
FAIL  A rerun of the committed params.yaml gives a growth rate this exercise doesn't accept. A rerun alone won't fix that. Compare the degrees with holdout.py, then decide.
FAIL  No commit message names the commit git bisect found, with at least its first 7 characters, and says why you chose this degree. Use the word degree and give your reason.
FAIL  The pre-commit hook is not installed in project/. Copy hooks/pre-commit into .git/hooks/. A clone does not bring hooks with it.
FAIL  No environment at envs/co2. Build it with the commands in step 1.
FAIL  remote/co2.git has no annotated tag v1.0. Use git tag -a v1.0, then git push origin v1.0.

0 of 8 parts pass.


## Why this CSV isn't in Git LFS

It's 39 KB, and Git handles that fine. Large files are different, because they make every clone heavy. Git LFS keeps them out of the repository, with commands such as `git lfs track "*.pt"` for checkpoints. LFS needs a server. In exercise 2 you'll use DVC, which doesn't.

## On your own project

Look at your last result. Is the `metrics.json` or table you shared the output of what's in Git right now? Add a check like `check_fresh.py`, install it as a hook, and write the lock today.

## About the data

Monthly mean CO2 at Mauna Loa, `data/co2_mm_mlo.csv`, 39 KB, unchanged from [NOAA Global Monitoring Laboratory](https://gml.noaa.gov/ccgg/trends/data.html). NOAA makes the data "freely available to the public and the scientific community" and asks for credit. The terms are in the file's header.

Cite it as: Dr. Xin Lan, NOAA/GML (gml.noaa.gov/ccgg/trends/) and Dr. Ralph Keeling, Scripps Institution of Oceanography (scrippsco2.ucsd.edu/).